# Microsoft Agent Framework

Built on Semantic Kernel's primitives, the Microsoft Agent Framework ships three orchestration patterns out of the box:

| Pattern | When to use it | Latency |
|---|---|---|
| **Sequential** | clear pipeline stages | sum of stages |
| **Concurrent** | N independent perspectives | max of N |
| **Group chat** | debate / consensus / review | unbounded (manager-driven) |

Picking the right pattern is itself a design decision.

## Canonical code (with `agent-framework` / `semantic-kernel` installed)

```python
from agent_framework import ChatAgent, SequentialWorkflow
from agent_framework.openai import OpenAIChatClient
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from task import search_corpus, fetch_paper, cite

client = OpenAIChatClient(model='gpt-4o-mini')

researcher = ChatAgent(
    name='Researcher',
    instructions='Find the relevant arxiv paper for the question. Output the arxiv_id and title only.',
    tools=[search_corpus, fetch_paper],
    chat_client=client,
)
summariser = ChatAgent(
    name='Summariser',
    instructions='Write a 2-sentence answer that cites the paper [arxiv_id]. Use `cite` to verify.',
    tools=[cite],
    chat_client=client,
)

workflow = SequentialWorkflow([researcher, summariser])
result = await workflow.run('What is RA-MoE?')
```

Concurrent and group-chat workflows are drop-in replacements for `SequentialWorkflow`.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / '03-agentic-frameworks'))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
from task import get_question
from eval import msaf_solve
trace = msaf_solve(get_question('q01'))
for step in trace.steps:
    print(f"{step.role}: {step.name or ''}  {step.output_summary or step.content or ''}"[:140])

## Tradeoffs

* **+ Three orchestration shapes built in** — no need to wire your own graph.
* **+ First-party Microsoft / Azure integration** if you're already in that ecosystem.
* **− Heavier dependency tree** than Pydantic AI or smolagents.
* **− Python docs are improving but lag the C# story.**